# Test on CaffeNet4096 and GoogleNet1024 Data

In [12]:
import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
from scipy.io import loadmat
featuresToUse = ["CaffeNet4096", "GoogleNet1024"] 
sourceDomainName = ['amazon'] #['caltech10','amazon','webcam']
targetDomainName = ['amazon'] #['caltech10','amazon','webcam']

tests = []
data_source = {}
data_target = {}

min_max_scaler = sklearn.preprocessing.MinMaxScaler()
# Collab
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[0],
                                                 "caltech10" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
S_data = [feat, labels]
S_nClass = len(np.unique(labels)) # nb de class in source data
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[1],
                                                 "amazon" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
T_data = [feat, labels]
T_nClass = len(np.unique(labels)) # nb de class in target data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1

In [6]:
from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference

In [13]:
results = []
numRepetitions = 10
nbtrain = 800
nbtest = 200
a = 0.4

prop_target_values_s = [0.01, 0.05, 0.1, 0.2, 0.4]
prop_target_values_p = [0.01, 0.05, 0.1, 0.2, 0.4]

XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1

XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1

for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    a = np.random.choice(np.arange(len(source)), math.ceil(0.7 * len(source)), replace=False)
    b = np.random.choice(np.arange(len(target)), math.ceil(0.7 * len(target)), replace=False)

    S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
    T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
    S = source.iloc[a, :].reset_index(drop=True)
    T = target.iloc[b, :].reset_index(drop=True)

    # =========================================================
    # UNSUPERVISED
    # =========================================================
     # COOT
    pure_source, pure_target, test_source, test_target = \
        discrete_unsupervised_coot(S, T, S_test, T_test)

    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })
    
    # JDCOOT
    pure_source, pure_target, test_source, test_target = \
        discrete_unsupervised_jdcoot(S, T, S_test, T_test)

    results.append({
        "repetition": repe,
        "recoding": "jdcoot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

   
import pandas as pd

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

df_summary
  

num repe : 1
Delta:       0.0457897 	 Loss:       1.9857736
Delta:       0.0595184 	 Loss:       1.6529402
Delta:       0.0548598 	 Loss:       1.4643111
Delta:       0.0453964 	 Loss:       1.4155604
Delta:       0.0351196 	 Loss:       1.4021765
Delta:       0.0266262 	 Loss:       1.3962759
Delta:       0.0237078 	 Loss:       1.3928607
Delta:       0.0221364 	 Loss:       1.3903178
Delta:       0.0208595 	 Loss:       1.3881691
Delta:       0.0179384 	 Loss:       1.3871043
Delta:       0.0153784 	 Loss:       1.3863319
Delta:       0.0172630 	 Loss:       1.3853919
Delta:       0.0163311 	 Loss:       1.3846756
Delta:       0.0154443 	 Loss:       1.3841790
Delta:       0.0153176 	 Loss:       1.3837845
Delta:       0.0145081 	 Loss:       1.3834240
Delta:       0.0125875 	 Loss:       1.3831766
Delta:       0.0117419 	 Loss:       1.3829772
Delta:       0.0127322 	 Loss:       1.3826721
Delta:       0.0143546 	 Loss:       1.3822381
Delta:       0.0132382 	 Loss:       1.3819110


,recoding,learning,prop_source,prop_target,pure_source_mean,pure_source_var,test_source_mean,test_source_var,pure_target_mean,pure_target_var,test_target_mean,test_target_var
0,coot,unsupervised,1,0,1.0,0.0,1.0,0.0,0.247094,0.017914,0.235192,0.021298
1,jdcoot,unsupervised,1,0,1.0,0.0,1.0,0.0,0.134426,0.000387,0.156794,0.001246


In [6]:
source

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X4087,X4088,X4089,X4090,X4091,X4092,X4093,X4094,X4095,Z
0,0.000000,2.505615,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,2.626899,...,0.000000,1.040452,0.000000,0.244105,0.538311,0.0,0.276814,0.000000,3.052249,0.0
1,0.000000,0.541579,0.313014,0.439709,0.000000,0.0,0.000000,0.000000,0.066556,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
2,0.790015,1.988702,0.000000,0.564894,0.799181,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.120850,0.154717,0.0,0.469833,0.000000,5.262370,0.0
3,0.617630,0.000000,0.000000,0.531292,0.000000,0.0,0.000000,0.063488,0.000000,2.252140,...,0.000000,2.184550,0.000000,4.910501,0.125535,0.0,0.112745,0.000000,0.000000,0.0
4,0.086308,0.000000,0.000000,0.000000,0.078548,0.0,0.000000,0.000010,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.817102,1.381422,0.0,0.000000,0.000000,0.951175,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,0.000000,0.022826,0.000000,0.000000,2.617168,0.0,0.000000,0.000000,0.459277,0.053653,...,0.137178,0.000000,0.000000,6.440595,0.000000,0.0,1.054335,0.000000,3.067745,9.0
1119,0.000000,0.000000,0.074741,0.000000,4.969587,0.0,0.000000,0.000000,0.036196,0.000000,...,0.255194,0.000000,1.990188,0.282888,1.212520,0.0,1.090388,0.448483,0.914598,9.0
1120,0.067069,0.115418,0.830895,0.242900,0.000000,0.0,1.496816,0.000000,0.000000,0.134041,...,0.209118,0.000000,3.301181,0.085582,1.132439,0.0,0.142505,5.160264,0.000000,9.0
1121,0.000000,0.004926,1.155382,0.000000,0.703217,0.0,0.000000,0.000000,0.000000,0.000000,...,0.128274,0.000000,1.349111,1.215771,0.055194,0.0,0.001579,4.097502,0.000000,9.0


In [54]:
##j'enregistre les données pour utiliser la fonction get_data
XtotS=S_data[0]
YtotS=S_data[1].reshape(-1,1)
dS=XtotS.shape[1]

XtotT=T_data[0]
YtotT=T_data[1].reshape(-1,1)
dS=XtotT.shape[1]

In [4]:
YtotS

array([[ 1],
       [ 1],
       [ 1],
       ...,
       [10],
       [10],
       [10]], shape=(1123, 1))

In [5]:
target

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X1015,X1016,X1017,X1018,X1019,X1020,X1021,X1022,X1023,Z
0,0.099118,0.027809,5.621949,1.360448,0.685656,0.002524,0.000000,0.222424,0.191003,0.000000,...,0.017318,6.049173,1.842010,3.035832,0.000000,0.000000,6.033271,4.444777,0.777563,0.0
1,0.347711,0.184287,0.748881,0.006219,0.925356,0.021182,0.120170,0.270272,0.511218,1.030571,...,2.210097,0.035351,1.422792,0.144325,0.022264,0.009212,0.867809,0.000000,0.000000,0.0
2,0.312158,0.130354,9.469339,1.738539,0.250605,0.536205,0.043808,0.001698,0.042926,0.177376,...,0.282821,3.069715,3.318993,1.846191,0.137990,0.002659,0.498049,2.357793,1.681124,0.0
3,1.520496,0.000000,2.717583,0.232378,0.380185,0.017633,0.000000,0.002723,0.000121,0.001088,...,1.243849,1.708731,1.281769,0.507668,0.000000,0.000000,0.000000,5.837334,0.576506,0.0
4,0.016194,0.004919,3.887881,0.263304,0.214002,0.005530,0.000000,0.381500,0.031129,0.254239,...,1.618137,3.108347,2.338883,4.205164,0.000000,0.000000,0.655401,0.516461,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,0.969091,0.000000,0.381402,1.017459,0.455186,0.036823,0.558800,0.000000,0.693394,1.862158,...,0.257680,0.000631,0.000000,4.676526,0.438856,0.000000,0.408622,0.000000,0.894359,9.0
1119,1.784889,0.000000,1.994400,0.001530,0.000000,0.000000,0.000000,0.000000,0.000000,0.085341,...,0.636262,0.000000,0.000000,0.085995,0.346864,0.000000,0.000000,0.042968,0.000000,9.0
1120,0.735216,0.022285,0.312883,0.071244,0.117068,0.104118,3.865505,0.000000,1.534048,0.003839,...,0.000755,0.000000,0.005801,3.084740,0.000241,1.087159,0.000000,0.111928,0.035476,9.0
1121,2.829518,0.000000,1.018245,0.000000,0.067400,0.000000,0.121775,0.000000,0.001408,0.783866,...,0.138958,0.000000,0.000000,0.435474,0.053507,0.000000,0.000000,0.000000,0.000000,9.0


In [40]:
import math

a = np.random.choice(np.arange(len(source)), math.ceil(0.7 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.7 * len(target)), replace=False)

S = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S.loc[:,'Z'] = S.loc[:, 'Z'] - 1
T.loc[:,'Z'] = T.loc[:, 'Z'] - 1
S

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X4087,X4088,X4089,X4090,X4091,X4092,X4093,X4094,X4095,Z
0,0.000000,0.541579,0.313014,0.439709,0.000000,0.000000,0.000000,0.000000,0.066556,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
1,0.790015,1.988702,0.000000,0.564894,0.799181,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.120850,0.154717,0.0,0.469833,0.000000,5.262370,0.0
2,0.617630,0.000000,0.000000,0.531292,0.000000,0.000000,0.000000,0.063488,0.000000,2.252140,...,0.000000,2.184550,0.000000,4.910501,0.125535,0.0,0.112745,0.000000,0.000000,0.0
3,0.000000,1.463537,5.183853,0.025951,0.000000,0.000000,0.000000,0.000000,0.000000,1.488524,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.122256,0.004240,0.0
4,0.000000,0.170888,0.000000,0.000000,0.000000,0.000000,0.063521,0.000000,0.000000,0.454374,...,0.000000,1.429933,0.554840,0.541800,1.083738,0.0,0.000000,0.000000,3.284321,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,0.000000,0.099487,0.000000,0.358152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.979591,0.615633,0.0,3.503209,3.392829,0.000000,9.0
332,0.000000,0.000000,1.070332,0.000000,0.075848,0.000000,0.000000,0.000000,4.890769,0.000000,...,0.086535,0.000000,0.000000,5.318273,1.588444,0.0,3.033021,0.421139,2.786675,9.0
333,0.162715,0.026648,1.700754,2.961777,2.158117,0.177779,0.055664,0.000000,0.918156,1.964597,...,2.096488,0.000000,2.597763,0.000000,0.000000,0.0,0.000000,7.352889,0.000000,9.0
334,0.000000,0.000000,1.976933,0.331685,1.350030,0.000000,0.000000,0.000000,1.066061,0.000000,...,0.000000,0.000000,1.976173,0.159130,0.404491,0.0,0.260178,3.789771,0.000000,9.0


In [6]:
S_test = source.iloc[a, :].reset_index(drop=True)
T_test = target.iloc[b, :].reset_index(drop=True)

## Performance function

In [21]:
from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
discrete_unsupervised_jdcoot( S, T, S_test, T_test)

Delta: 0.021263271369738355 	  Loss: 2.162925540821128 	 Accuracy: 0.23333333333333334
Delta: 0.020000443417459525 	  Loss: 2.077011289336405 	 Accuracy: 0.31333333333333335
Delta: 0.019055079567157088 	  Loss: 1.9422504985775286 	 Accuracy: 0.25
Delta: 0.01773476045373289 	  Loss: 1.802982249875017 	 Accuracy: 0.21
Delta: 0.015708656033495066 	  Loss: 1.7122225735868104 	 Accuracy: 0.24
Delta: 0.01304045635073084 	  Loss: 1.668447388605732 	 Accuracy: 0.25666666666666665
Delta: 0.011122203384496373 	  Loss: 1.6476508289899934 	 Accuracy: 0.26


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.009831922530629051 	  Loss: 1.6385706561676203 	 Accuracy: 0.27
Delta: 0.008779914015812471 	  Loss: 1.6345855889900567 	 Accuracy: 0.27666666666666667
Delta: 0.008487088776980843 	  Loss: 1.6318112117679633 	 Accuracy: 0.28


(1.0, np.float64(0.28), 1.0, np.float64(0.29))

In [22]:
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
discrete_semisupervised_jdcoot( S, T, S_test, T_test, prop_source=0.2)

Delta: 0.024053524611996183 	 Loss: 1.6337119828289188 	 Accuracy: 0.7925925925925926
Delta: 0.018851986726779076 	 Loss: 1.5343537041160054 	 Accuracy: 0.8333333333333334


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.011137290831537711 	 Loss: 1.512377304401441 	 Accuracy: 0.8814814814814815
Delta: 0.008898383276182921 	 Loss: 1.5038220107080624 	 Accuracy: 0.8888888888888888
Delta: 0.00752849928720386 	 Loss: 1.4995465365787748 	 Accuracy: 0.8888888888888888
Delta: 0.006613635026971421 	 Loss: 1.497288857462081 	 Accuracy: 0.8925925925925926
Delta: 0.006611524726305552 	 Loss: 1.4955469513757307 	 Accuracy: 0.8962962962962963
Delta: 0.0073783125194536985 	 Loss: 1.4934260133941064 	 Accuracy: 0.8962962962962963
Delta: 0.0056836282059234055 	 Loss: 1.4927349335644085 	 Accuracy: 0.8925925925925926
Delta: 0.004241687632779049 	 Loss: 1.4919720199010182 	 Accuracy: 0.9037037037037037


(1.0, np.float64(0.9037037037037037), 1.0, np.float64(0.95))

In [23]:
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
discrete_partial_jdcoot( S, T, S_test, T_test, prop_source=0.2, prop_target=0.2)

Delta: 0.025503346276540908 	  Loss: 1.5519063252155973 	 Accuracy: 0.8458333333333333
Delta: 0.019436473734944684 	  Loss: 1.5308519522368509 	 Accuracy: 0.8333333333333334
Delta: 0.01559955183990858 	  Loss: 1.553659682250378 	 Accuracy: 0.7333333333333333


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.012302633000622535 	  Loss: 1.5842327411555606 	 Accuracy: 0.6833333333333333
Delta: 0.011518121632367699 	  Loss: 1.6134749115504106 	 Accuracy: 0.6458333333333334
Delta: 0.010455461558006807 	  Loss: 1.6352861741176763 	 Accuracy: 0.6125
Delta: 0.007988561068197716 	  Loss: 1.6525283440803118 	 Accuracy: 0.5458333333333333
Delta: 0.0062719520370890566 	  Loss: 1.6629730226830874 	 Accuracy: 0.475
Delta: 0.0047306425033502185 	  Loss: 1.6682945724370266 	 Accuracy: 0.44583333333333336
Delta: 0.0044830446044753156 	  Loss: 1.6714345977308909 	 Accuracy: 0.3541666666666667


(np.float64(0.3125),
 np.float64(0.3541666666666667),
 np.float64(0.6),
 np.float64(0.72))

In [24]:
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
discrete_unsupervised_coot( S, T, S_test, T_test)


Delta:       0.0156694 	 Loss:       2.2046072
Delta:       0.0211672 	 Loss:       2.1673489
Delta:       0.0188271 	 Loss:       2.1294881


KeyboardInterrupt: 

In [16]:
# get test set : save examples (nbtest per class) 
# take the last ones so that it's always the same set and we can compare results
# get train set with random
import numpy as np
from random import shuffle
from sklearn.preprocessing import OneHotEncoder as onehot
from sklearn.model_selection import train_test_split

def get_data(x, y, nbtrain, nbtest, nseed):
 
    y = y.ravel()
    n = x.shape[0]

    if nbtrain + nbtest > n:
        raise ValueError("nbtrain + nbtest exceeds total number of samples")

    np.random.seed(nseed)
    idx = np.random.permutation(n)

    train_idx = idx[:nbtrain]
    test_idx = idx[nbtrain:nbtrain + nbtest]

    xtrain = x[train_idx]
    ytrain = y[train_idx]

    xtest = x[test_idx]
    ytest = y[test_idx]

    return xtrain, ytrain, xtest, ytest


def get_data_2(x,y,nbtrain,nbtest,nseed):
    
    xtrain=np.zeros((0,x.shape[1]))
    ytrain=np.zeros((0))
    xtest=np.zeros((0,x.shape[1]))
    ytest=np.zeros((0))
    print(np.unique(y))
    for i in np.unique(y):
        xi=x[y.ravel()==i,:] # tous les exemples de la classe i

        if len(np.argwhere(y==i)) >= nbtrain+nbtest :
            ni = xi.shape[0] - nbtest    # nb d'exemples de la classe i pouvant servir lors de l'apprentissage
            np.random.seed(nseed)
            idx=np.random.permutation(ni)
          #print(idx[:nbtrain])
        
            xtrain=np.concatenate((xtrain,xi[idx[:nbtrain],:]),0)
            ytrain=np.concatenate((ytrain,i*np.ones(nbtrain)))

            xtest=np.concatenate((xtest,xi[ni:]),0)
            ytest=np.concatenate((ytest,i*np.ones(nbtest)))
 
        else:
            np.random.seed(nseed)
            idx=np.random.permutation(xi.shape[0])
        
            xtrain=np.concatenate((xtrain,xi[idx[:nbtrain]]),0)
            ytrain=np.concatenate((ytrain,i*np.ones(nbtrain)))
 
            xtest=np.concatenate((xtest,xi[idx[nbtrain:]]),0)
            ytest=np.concatenate((ytest,i*np.ones(xi.shape[0]-nbtrain)))
        
    return xtrain,ytrain,xtest,ytest


# semi-supervison
# get nsamples labelled examples for each class and one-hot-encode those labels
# nclass = number of classes in the dataset
# nsamples : number of labelles samples per class
# noLabClass : classes chosen to remain unlabelled
def get_labels(y,nsamples,nclass=10,noLabClass=[]):
    Y = np.zeros((len(y),nclass))
    for c in np.unique(y):
        if c not in noLabClass :
            idx = np.where(y==c)[0]
            Y[idx[:nsamples],int(c)]=1
    return Y

def generateSubset(X, Y, nPerClass):
    idx = []
    for c in np.unique(Y):
        idxClass = np.argwhere(Y == c).ravel()
        shuffle(idxClass)
        idx.extend(idxClass[0:min(nPerClass, len(idxClass))])
    return (X[idx, :], Y[idx])



In [9]:
from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference
from jdcoot.models.continuous_unsupervised_jdcoot import continuous_unsupervised_jdcoot
from jdcoot.models.continuous_unsupervised_coot import continuous_unsupervised_coot

## alpha

In [52]:
from jdcoot.utils import xcolumns, discrete_classifier, discrete_accuracy
from jdcoot.coot import init_matrix_np
from jdcoot.losses import loss_crossentropy2
from tf_keras.utils import to_categorical


def one_hot(y, nClass):
    return to_categorical(y, num_classes=nClass)


def one_cold(z_encoded):
    return np.argmax(z_encoded, axis=1)

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1

classes = np.union1d(np.unique(source.Z), np.unique(target.Z))
nClass = len(classes)

x_source = source.loc[:, xcolumns(source)].values
z_source = source.Z.values

x_target = target.loc[:, xcolumns(target)].values
z_target = target.Z.values

z_target_u= np.unique(z_target)


max_diff2 = max(
    (x_source.max() - x_target.min())**2,
    (x_source.min() - x_target.max())**2
)
max_diff2
# Calcul de toutes les distances euclidiennes

# Distance maximale
#max_distance = np.max(dist_matrix^2)
#print("Max distance:", max_distance)

z1 = np.arange(10)
z2 = np.arange(10)
n_class = 10
Z1 = one_hot(z1, n_class)  # (10, 10)
Z2 = one_hot(z2, n_class)
fcost = np.max(loss_crossentropy2(Z1, Z2))

alpha = fcost/max_diff2
alpha

np.float64(0.044981382834838365)

In [ ]:
discrete_unsupervised_jdcoot( S, T, S_test, T_test,alpha=a)

## Data labellisation impact

In [53]:
import numpy as np
nbtrain=80
nbtest=20
XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1
XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1

XS1,yS1,XStest,yStest = get_data(XtotS,YtotS,nbtrain,nbtest,repe)
XT1,yT1,XTtest,yTtest = get_data(XtotT,YtotT,nbtrain,nbtest,repe)

    #XS, yS = generateSubset(XS1, yS1, perClassSource)
    #XT, yT = generateSubset(XT1, yT1, perClassSource)
S = pd.DataFrame(np.c_[XS1, yS1],
                     columns=['X' + str(i) for i in range(XS1.shape[1])] + ['Z'])
T = pd.DataFrame(np.c_[XT1, yT1],
                     columns=['X' + str(i) for i in range(XT1.shape[1])] + ['Z'])
S_test = pd.DataFrame(np.c_[XStest, yStest],
                          columns=['X' + str(i) for i in range(XStest.shape[1])] + ['Z'])
T_test = pd.DataFrame(np.c_[XTtest, yTtest],
                          columns=['X' + str(i) for i in range(XTtest.shape[1])] + ['Z'])


# Exemple de valeurs à tester pour alpha
alpha_values = np.linspace(0, 1, 11)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique

for a in alpha_values:

    pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(source, target, source, target, alpha=a)

    score = test_target  
    
    if score > best_score:
        best_score = score
        best_alpha = a

print("Meilleur alpha :", best_alpha)
print("Score associé :", best_score)

[0 1 2 3 4 5 6 7 8 9]
[0 1 2 3 4 5 6 7 8 9]
Delta: 0.020070853769768717 	  Loss: 2.1992692919258294 	 Accuracy: 0.13446126447016918
Delta: 0.013934807273575637 	  Loss: 2.1991286543979776 	 Accuracy: 0.13446126447016918
Delta: 0.017891672496933107 	  Loss: 2.1988780462113287 	 Accuracy: 0.13446126447016918
Delta: 0.019324073522942672 	  Loss: 2.1981585946912228 	 Accuracy: 0.13446126447016918
Delta: 0.016070939707578908 	  Loss: 2.197205850945167 	 Accuracy: 0.13446126447016918
Delta: 0.014223091316730711 	  Loss: 2.195257141834942 	 Accuracy: 0.12644701691896706
Delta: 0.01283025365292386 	  Loss: 2.1927323041734317 	 Accuracy: 0.12377560106856635
Delta: 0.013382902991117718 	  Loss: 2.1888384137377126 	 Accuracy: 0.12021371326803205
Delta: 0.014149537744283089 	  Loss: 2.1850826878329803 	 Accuracy: 0.11308993766696349
Delta: 0.015192445211071383 	  Loss: 2.1821262492144773 	 Accuracy: 0.11130899376669635
Delta: 0.01959983195849899 	  Loss: 2.1945266048663825 	 Accuracy: 0.1344612644

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.008628309091628815 	  Loss: 1.6785895818255219 	 Accuracy: 0.5351736420302761
Delta: 0.020446624782429333 	  Loss: 2.1711228134546348 	 Accuracy: 0.26001780943900266
Delta: 0.017872224851115736 	  Loss: 2.1180715236712526 	 Accuracy: 0.3268032056990205
Delta: 0.01617891022742014 	  Loss: 2.0399759756427 	 Accuracy: 0.43544078361531613
Delta: 0.016106766009112497 	  Loss: 1.93168343221797 	 Accuracy: 0.42030276046304543
Delta: 0.014789515908552351 	  Loss: 1.8255870000728045 	 Accuracy: 0.42920747996438113
Delta: 0.013566441709605468 	  Loss: 1.7577499612888186 	 Accuracy: 0.41406945681211044
Delta: 0.012185246632321416 	  Loss: 1.7174230921505471 	 Accuracy: 0.40872662511130897
Delta: 0.009961890774380144 	  Loss: 1.6930335750589849 	 Accuracy: 0.40160284951024044
Delta: 0.008332848662934144 	  Loss: 1.6801426975241838 	 Accuracy: 0.40694568121104185


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.006645980171424089 	  Loss: 1.6733689802408707 	 Accuracy: 0.39982190560997327
Delta: 0.020696340959246218 	  Loss: 2.1649618238775203 	 Accuracy: 0.24577025823686555
Delta: 0.018011837234799877 	  Loss: 2.100742260483469 	 Accuracy: 0.27070347284060553
Delta: 0.016704976735204036 	  Loss: 2.0195044108722096 	 Accuracy: 0.3472840605520926
Delta: 0.01587392222370451 	  Loss: 1.9398575198820658 	 Accuracy: 0.38201246660730187
Delta: 0.014716509656527831 	  Loss: 1.8843051742426882 	 Accuracy: 0.3633125556544969
Delta: 0.015190502297568756 	  Loss: 1.8341293258313842 	 Accuracy: 0.3348174532502226
Delta: 0.013728420276592301 	  Loss: 1.7807463025776067 	 Accuracy: 0.3107747105966162
Delta: 0.011405375626621717 	  Loss: 1.7444452093654124 	 Accuracy: 0.3081032947462155
Delta: 0.009265589754914589 	  Loss: 1.723047959431587 	 Accuracy: 0.3170080142475512
Delta: 0.008361708987431547 	  Loss: 1.708517045958742 	 Accuracy: 0.31522707034728403
Delta: 0.020760461047763815 	  Loss: 2.158

...

Delta: 0.005653905899521092 	  Loss: 1.6881844216844801 	 Accuracy: 0.3250222617987533
Delta: 0.0210226161289102 	  Loss: 2.145339081717418 	 Accuracy: 0.2965271593944791
Delta: 0.018523131414265707 	  Loss: 2.042606086725181 	 Accuracy: 0.3312555654496883
Delta: 0.016672562928165553 	  Loss: 1.9310147223854972 	 Accuracy: 0.2858414959928762
Delta: 0.01592465316449665 	  Loss: 1.839633810123288 	 Accuracy: 0.2840605520926091
Delta: 0.014806664067142122 	  Loss: 1.76711422904787 	 Accuracy: 0.2867319679430098
Delta: 0.012373676507967698 	  Loss: 1.7257111944807748 	 Accuracy: 0.29118432769367764
Delta: 0.010299589045498405 	  Loss: 1.7060120664237801 	 Accuracy: 0.2947462154942119
Delta: 0.0088734375497298 	  Loss: 1.6952734304969406 	 Accuracy: 0.2929652715939448


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.0064467002362850875 	  Loss: 1.6916388376151006 	 Accuracy: 0.2840605520926091
Delta: 0.005411496368861437 	  Loss: 1.6901905101803045 	 Accuracy: 0.2902938557435441
Delta: 0.021145035879407744 	  Loss: 2.137406589932832 	 Accuracy: 0.2920747996438112
Delta: 0.01852936073258541 	  Loss: 2.016796308944894 	 Accuracy: 0.3535173642030276
Delta: 0.017274207787774144 	  Loss: 1.8954118367053785 	 Accuracy: 0.3312555654496883
Delta: 0.016158244914877225 	  Loss: 1.7936688233970886 	 Accuracy: 0.3223508459483526
Delta: 0.01417415502039207 	  Loss: 1.7334390766564711 	 Accuracy: 0.333926981300089
Delta: 0.011598835791677429 	  Loss: 1.6988934854018087 	 Accuracy: 0.3285841495992876
Delta: 0.009448213339043604 	  Loss: 1.6804655755655846 	 Accuracy: 0.3348174532502226


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.007832331569469846 	  Loss: 1.6730126709240536 	 Accuracy: 0.3294746215494212
Delta: 0.00603792501236919 	  Loss: 1.6702879658397736 	 Accuracy: 0.3268032056990205
Delta: 0.004625327850154655 	  Loss: 1.6693480294792362 	 Accuracy: 0.3321460373998219
Meilleur alpha : 0.4
Score associé : 0.5351736420302761


In [16]:
results = []
numRepetitions = 10
nbtrain=80
nbtest=20
a= 0.4
n_source = len(source.Z)
n_target = len(target.Z)
prop_target_values_s = [0.01,0.05,0.1,0.2,0.4]
prop_target_values_p = [0.01,0.05,0.1,0.2,0.4]
XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1
XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1
XtotS.shape
repe=1

In [7]:
XS1,yS1,XStest,yStest = get_data(XtotS,YtotS,nbtrain,nbtest,repe)
XT1,yT1,XTtest,yTtest = get_data(XtotT,YtotT,nbtrain,nbtest,repe)

    #XS, yS = generateSubset(XS1, yS1, perClassSource)
    #XT, yT = generateSubset(XT1, yT1, perClassSource)
S = pd.DataFrame(np.c_[XS1, yS1],
                     columns=['X' + str(i) for i in range(XS1.shape[1])] + ['Z'])
T = pd.DataFrame(np.c_[XT1, yT1],
                     columns=['X' + str(i) for i in range(XT1.shape[1])] + ['Z'])
S_test = pd.DataFrame(np.c_[XStest, yStest],
                          columns=['X' + str(i) for i in range(XStest.shape[1])] + ['Z'])
T_test = pd.DataFrame(np.c_[XTtest, yTtest],
                          columns=['X' + str(i) for i in range(XTtest.shape[1])] + ['Z'])


In [13]:
T['Z'].value_counts()
classes = np.union1d(np.unique(source.Z), np.unique(target.Z))
nClass = len(classes)
nClass
from jdcoot.utils import xcolumns, discrete_classifier, discrete_accuracy

x_source = source.loc[:, xcolumns(source)].values
z_source = source.Z.values

x_target = target.loc[:, xcolumns(target)].values
z_target = target.Z.values

clf = discrete_classifier(target, "relu", "softmax", nClass)


InternalError: cudaSetDevice() on GPU:0 failed. Status: out of memory

In [9]:
discrete_unsupervised_jdcoot(S,T, S_test, T_test,alpha=a)

InternalError: cudaSetDevice() on GPU:0 failed. Status: out of memory

In [24]:
discrete_unsupervised_coot( S, T, S_test, T_test,alpha=a)

Delta:       0.0156361 	 Loss:       2.2010207
Delta:       0.0208233 	 Loss:       2.1663747
Delta:       0.0182178 	 Loss:       2.1294713
Delta:       0.0162790 	 Loss:       2.0961136
Delta:       0.0131377 	 Loss:       2.0713907
Delta:       0.0080867 	 Loss:       2.0659345
Delta:       0.0057522 	 Loss:       2.0649909
Delta:       0.0054997 	 Loss:       2.0646522
Delta:       0.0051974 	 Loss:       2.0643182
Delta:       0.0050661 	 Loss:       2.0640708
Delta:       0.0043271 	 Loss:       2.0638897
Delta:       0.0034496 	 Loss:       2.0636709
Delta:       0.0026599 	 Loss:       2.0635733
Delta:       0.0031707 	 Loss:       2.0635515
Delta:       0.0037098 	 Loss:       2.0635059
Delta:       0.0041243 	 Loss:       2.0634456
Delta:       0.0049578 	 Loss:       2.0633264
Delta:       0.0052853 	 Loss:       2.0631646
Delta:       0.0051491 	 Loss:       2.0629489
Delta:       0.0053762 	 Loss:       2.0627186
Delta:       0.0048121 	 Loss:       2.0625464
Delta:       

(1.0, np.float64(0.56125), 1.0, np.float64(0.5705521472392638))

In [63]:
target.shape

(1123, 1025)

In [ ]:
import math

results = []
numRepetitions = 1
nbtrain=800
nbtest=200
a= 0.4
repe=1
n_source = len(source.Z)
n_target = len(target.Z)
prop_target_values_s = [0.01,0.05,0.1,0.2,0.4]
prop_target_values_p = [0.01,0.05,0.1,0.2,0.4]
XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1
XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1


#for repe in range(numRepetitions):
#    print('num repe :', repe + 1)
XS1,yS1,XStest,yStest = get_data(XtotS,YtotS,nbtrain,nbtest,repe)
XT1,yT1,XTtest,yTtest = get_data(XtotT,YtotT,nbtrain,nbtest,repe)

    #XS, yS = generateSubset(XS1, yS1, perClassSource)
    #XT, yT = generateSubset(XT1, yT1, perClassSource)
S = pd.DataFrame(np.c_[XS1, yS1],
                     columns=['X' + str(i) for i in range(XS1.shape[1])] + ['Z'])
T = pd.DataFrame(np.c_[XT1, yT1],
                     columns=['X' + str(i) for i in range(XT1.shape[1])] + ['Z'])
S_test = pd.DataFrame(np.c_[XStest, yStest],
                          columns=['X' + str(i) for i in range(XStest.shape[1])] + ['Z'])
T_test = pd.DataFrame(np.c_[XTtest, yTtest],
                          columns=['X' + str(i) for i in range(XTtest.shape[1])] + ['Z'])

    # =========================================================
    # UNSUPERVISED
    # =========================================================
#for recoding, func in [
#        ("jdcoot", discrete_unsupervised_jdcoot),
 #       ("coot", discrete_unsupervised_coot),
 #   ]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T, S, T,alpha=a)

test_target


Delta: 0.020604676069039197 	  Loss: 2.1899771523466036 	 Accuracy: 0.24
Delta: 0.01804443017389411 	  Loss: 2.1461733353112944 	 Accuracy: 0.3425
Delta: 0.01659657501207717 	  Loss: 2.07914864052573 	 Accuracy: 0.37375
Delta: 0.015982322902591748 	  Loss: 1.9802047233556392 	 Accuracy: 0.42375
Delta: 0.014701896401629503 	  Loss: 1.8744333024682578 	 Accuracy: 0.4125
Delta: 0.013089507232964273 	  Loss: 1.792800133240287 	 Accuracy: 0.4175
Delta: 0.012079010558138816 	  Loss: 1.738908967152384 	 Accuracy: 0.42375
Delta: 0.011021812067774339 	  Loss: 1.7065118082858717 	 Accuracy: 0.4175
Delta: 0.00920255247274257 	  Loss: 1.6899054400003721 	 Accuracy: 0.42625
Delta: 0.007746608954590427 	  Loss: 1.6831068115604984 	 Accuracy: 0.43


np.float64(0.43)

In [12]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T, S, T,alpha=a)

test_target

Delta: 0.020568756714356855 	  Loss: 2.1899736784578447 	 Accuracy: 0.245
Delta: 0.01791652927670536 	  Loss: 2.1458999780142953 	 Accuracy: 0.34
Delta: 0.01672161443362766 	  Loss: 2.079870358259678 	 Accuracy: 0.36125
Delta: 0.015192863173359158 	  Loss: 1.998424922083945 	 Accuracy: 0.365
Delta: 0.013810260779751232 	  Loss: 1.9256867863685372 	 Accuracy: 0.4075
Delta: 0.01251125382340719 	  Loss: 1.8686862672692834 	 Accuracy: 0.3925
Delta: 0.011587098355555648 	  Loss: 1.8272850185496812 	 Accuracy: 0.415
Delta: 0.012273650138915706 	  Loss: 1.7936422427557575 	 Accuracy: 0.4075
Delta: 0.01173483828286506 	  Loss: 1.768925924429328 	 Accuracy: 0.40125
Delta: 0.010004606232456283 	  Loss: 1.754190308943675 	 Accuracy: 0.3875


np.float64(0.3875)

In [26]:
results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "unsupervised",
            "prop_source": 1,
            "prop_target": 0,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

In [15]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_coot(S, T, S, T)



Delta:       0.0156422 	 Loss:       2.2021110
Delta:       0.0208768 	 Loss:       2.1691646
Delta:       0.0177753 	 Loss:       2.1349569
Delta:       0.0143025 	 Loss:       2.1204074
Delta:       0.0134693 	 Loss:       2.1127211
Delta:       0.0137265 	 Loss:       2.1037610
Delta:       0.0131059 	 Loss:       2.0941868
Delta:       0.0108799 	 Loss:       2.0876489
Delta:       0.0082359 	 Loss:       2.0847121
Delta:       0.0070575 	 Loss:       2.0836706
Delta:       0.0048615 	 Loss:       2.0831098
Delta:       0.0036876 	 Loss:       2.0830085
Delta:       0.0014327 	 Loss:       2.0829536
Delta:       0.0016584 	 Loss:       2.0829508
Delta:       0.0004895 	 Loss:       2.0829506
Delta:       0.0007728 	 Loss:       2.0829506
converged at iter  15


In [16]:
test_target

np.float64(0.25333333333333335)

In [28]:
df_results = pd.DataFrame(results)
df_results
df_results.to_excel("results_mean.xlsx", index=False)

In [67]:
source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1

a=np.random.choice(np.arange(len(source.index)),math.ceil(0.7*len(source.index)),replace=False)
b=np.random.choice(np.arange(len(target.index)),math.ceil(0.7*len(target.index)),replace=False)
S_test=source.iloc[a,:].reset_index(drop=True)
T_test=target.iloc[b,:].reset_index(drop=True)

S=source.iloc[np.setdiff1d(np.arange(len(source.index)),a),:].reset_index(drop=True)
T=target.iloc[np.setdiff1d(np.arange(len(target.index)),b),:].reset_index(drop=True)
T_test.shape

(787, 1025)

In [47]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T,  S_test, T_test, alpha=0.66)


Delta: 0.02162954753151638 	  Loss: 2.2064435525074266 	 Accuracy: 0.16964285714285715
Delta: 0.0203060062042208 	  Loss: 2.0866812993621506 	 Accuracy: 0.27380952380952384
Delta: 0.019339676073705025 	  Loss: 1.921979274273892 	 Accuracy: 0.3333333333333333
Delta: 0.017074717446758987 	  Loss: 1.7897524394344986 	 Accuracy: 0.3392857142857143
Delta: 0.014697332874395607 	  Loss: 1.7172567367630043 	 Accuracy: 0.3244047619047619
Delta: 0.012540406545963776 	  Loss: 1.684557033197894 	 Accuracy: 0.31547619047619047
Delta: 0.010926340996103904 	  Loss: 1.6708734630853834 	 Accuracy: 0.3005952380952381
Delta: 0.00981756940332448 	  Loss: 1.6651487778450003 	 Accuracy: 0.30654761904761907
Delta: 0.008811436298066728 	  Loss: 1.66398045165981 	 Accuracy: 0.30357142857142855


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.007396354431511463 	  Loss: 1.6635911540376345 	 Accuracy: 0.30357142857142855


In [20]:
import math
results = []
numRepetitions = 10
a = 0.5

prop_target_values_s = [0.01, 0.05, 0.1, 0.2, 0.4]
prop_target_values_p = [0.01, 0.05, 0.1, 0.2, 0.4]



source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1

for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    a=np.random.choice(np.arange(len(source.index)),math.ceil(0.7*len(source.index)),replace=False)
    b=np.random.choice(np.arange(len(target.index)),math.ceil(0.7*len(target.index)),replace=False)
    S=source.iloc[a,:].reset_index(drop=True)
    T=target.iloc[b,:].reset_index(drop=True)

    S_test=source.iloc[np.setdiff1d(np.arange(len(source.index)),a),:].reset_index(drop=True)
    T_test=target.iloc[np.setdiff1d(np.arange(len(target.index)),b),:].reset_index(drop=True)
    # =========================================================
    # UNSUPERVISED
    # =========================================================

    # COOT (sans alpha)
    pure_source, pure_target, test_source, test_target = discrete_unsupervised_coot(S, T, S_test, T_test)

    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

    # JDCOOT
    pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T,  S_test, T_test, alpha = 0.4)

    results.append({
        "repetition": repe,
        "recoding": "jdcoot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

num repe : 1
Delta:       0.0156359 	 Loss:       2.2044734
Delta:       0.0209840 	 Loss:       2.1662393
Delta:       0.0185675 	 Loss:       2.1072568
Delta:       0.0146230 	 Loss:       2.0754322
Delta:       0.0112744 	 Loss:       2.0663307
Delta:       0.0085521 	 Loss:       2.0641118
Delta:       0.0071566 	 Loss:       2.0633101
Delta:       0.0063113 	 Loss:       2.0629898
Delta:       0.0058049 	 Loss:       2.0627484
Delta:       0.0047601 	 Loss:       2.0626239
Delta: 0.020483176888626983 	  Loss: 2.1909931176359447 	 Accuracy: 0.10581222056631892
Delta: 0.016912573173977947 	  Loss: 2.1694329823388045 	 Accuracy: 0.15648286140089418
Delta: 0.013698297827186042 	  Loss: 2.150870650525441 	 Accuracy: 0.17585692995529062
Delta: 0.012296416885201035 	  Loss: 2.137754422014215 	 Accuracy: 0.2488822652757079
Delta: 0.010050267048937023 	  Loss: 2.130399160342897 	 Accuracy: 0.29806259314456035
Delta: 0.00876778297624606 	  Loss: 2.1260583554748225 	 Accuracy: 0.309985096870

KeyboardInterrupt: 

In [ ]:
 # =========================================================
    # SEMI-SUPERVISED
    # =========================================================
    for prop_target in prop_target_values_s:

        # JDCOOT
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_jdcoot(
                S, T, S_test, T_test,
                alpha=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # COOT
        pure_source, pure_target, test_source, test_target = \
            discrete_semisupervised_coot(
                S, T, S_test, T_test,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "coot",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # Reference
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_reference(
                S, T, S_test, T_test,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "reference",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

    # =========================================================
    # PARTIAL
    # =========================================================
    for prop_target in prop_target_values_p:

        # JDCOOT
        pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot(
                S, T, S_test, T_test,
                alpha=0.5,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # COOT
        pure_source, pure_target, test_source, test_target = discrete_partial_coot(
                S, T, S_test, T_test,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "coot",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # Reference
        pure_source, pure_target, test_source, test_target = discrete_partial_reference(
                S, T, S_test, T_test,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "reference",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })    
 

In [77]:
df_summary

,recoding,learning,prop_source,prop_target,pure_source_mean,pure_source_var,test_source_mean,test_source_var,pure_target_mean,pure_target_var,test_target_mean,test_target_var
0,coot,unsupervised,1,0,1.0,0.0,1.0,0.0,0.405591,0.019907,0.414881,0.022938
1,jdcoot,unsupervised,1,0,1.0,0.0,1.0,0.0,0.317662,0.031219,0.325595,0.030766


In [ ]:
results = []
numRepetitions = 10
nbtrain = 80
nbtest = 20
a = 0.01

prop_target_values_s = [0.01, 0.05, 0.1, 0.2, 0.4]
prop_target_values_p = [0.01, 0.05, 0.1, 0.2, 0.4]

XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1

XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1

for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    XS1, yS1, XStest, yStest = get_data_2(XtotS, YtotS, nbtrain, nbtest, repe)
    XT1, yT1, XTtest, yTtest = get_data_2(XtotT, YtotT, nbtrain, nbtest, repe)

    S = pd.DataFrame(
        np.c_[XS1, yS1],
        columns=[f"X{i}" for i in range(XS1.shape[1])] + ["Z"]
    )

    T = pd.DataFrame(
        np.c_[XT1, yT1],
        columns=[f"X{i}" for i in range(XT1.shape[1])] + ["Z"]
    )

    S_test = pd.DataFrame(
        np.c_[XStest, yStest],
        columns=[f"X{i}" for i in range(XStest.shape[1])] + ["Z"]
    )

    T_test = pd.DataFrame(
        np.c_[XTtest, yTtest],
        columns=[f"X{i}" for i in range(XTtest.shape[1])] + ["Z"]
    )

    # =========================================================
    # UNSUPERVISED
    # =========================================================

    # JDCOOT
    pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T, S, T, alpha=a)

    results.append({
        "repetition": repe,
        "recoding": "jdcoot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

    # COOT (sans alpha)
    pure_source, pure_target, test_source, test_target = discrete_unsupervised_coot(S, T, S, T)

    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

 
import pandas as pd

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)
df_summary


num repe : 1
[0 1 2 3 4 5 6 7 8 9]
[0 1 2 3 4 5 6 7 8 9]
Delta: 0.0630407778513002 	  Loss: 1.8134424277156618 	 Accuracy: 0.1475
Delta: 0.051077024143968155 	  Loss: 1.7653979809586575 	 Accuracy: 0.145
Delta: 0.04172664749744727 	  Loss: 1.7461373861218679 	 Accuracy: 0.145
Delta: 0.03343335782028892 	  Loss: 1.7374525440389796 	 Accuracy: 0.145
Delta: 0.02559224042786981 	  Loss: 1.7350510020918537 	 Accuracy: 0.145
Delta: 0.025615296761164745 	  Loss: 1.732599169907346 	 Accuracy: 0.145
Delta: 0.02056974390388626 	  Loss: 1.7317195093896662 	 Accuracy: 0.145
Delta: 0.01872108461966716 	  Loss: 1.7311238297191918 	 Accuracy: 0.145
Delta: 0.013198731785563807 	  Loss: 1.7310184241670092 	 Accuracy: 0.145
Delta: 0.01322770690255867 	  Loss: 1.730984957432514 	 Accuracy: 0.145
Delta:       0.0509506 	 Loss:       1.9609350
Delta:       0.0670170 	 Loss:       1.5919715
Delta:       0.0585432 	 Loss:       1.4756371
Delta:       0.0493188 	 Loss:       1.4333216
Delta:       0.0484531 	

In [ ]:
   # =========================================================
    # SEMI-SUPERVISED
    # =========================================================
    for prop_target in prop_target_values_s:

        # JDCOOT
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_jdcoot(
                S, T, S_test, T_test,
                alpha=a,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # COOT
        pure_source, pure_target, test_source, test_target = \
            discrete_semisupervised_coot(
                S, T, S_test, T_test,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "coot",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # Reference
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_reference(
                S, T, S_test, T_test,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "reference",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

    # =========================================================
    # PARTIAL
    # =========================================================
    for prop_target in prop_target_values_p:

        # JDCOOT
        pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot(
                S, T, S_test, T_test,
                alpha=a,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # COOT
        pure_source, pure_target, test_source, test_target = discrete_partial_coot(
                S, T, S_test, T_test,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "coot",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # Reference
        pure_source, pure_target, test_source, test_target = discrete_partial_reference(
                S, T, S_test, T_test,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "reference",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })


In [30]:
df_results = pd.DataFrame(results)
df_results

,repetition,recoding,learning,prop_source,prop_target,pure_source,test_source,pure_target,test_target
0,0,jdcoot,unsupervised,1.0,0.00,1.0000,1.000000,0.525000,0.564417
1,0,coot,unsupervised,1.0,0.00,1.0000,1.000000,0.565000,0.558282
2,0,jdcoot,semisupervised,1.0,0.01,1.0000,1.000000,0.277778,0.300613
3,0,coot,semisupervised,1.0,0.01,1.0000,1.000000,0.898990,0.815951
4,0,reference,semisupervised,1.0,0.01,1.0000,1.000000,0.447853,0.491162
5,0,jdcoot,semisupervised,1.0,0.05,1.0000,1.000000,0.819737,0.920245
6,0,coot,semisupervised,1.0,0.05,1.0000,1.000000,0.881579,0.797546
7,0,reference,semisupervised,1.0,0.05,1.0000,1.000000,0.901840,0.821053
8,0,jdcoot,semisupervised,1.0,0.10,1.0000,1.000000,0.911111,0.957055
9,0,coot,semisupervised,1.0,0.10,1.0000,1.000000,0.912500,0.730061


In [62]:
print(df_results)
df_summary.to_excel("results_mean.xlsx", index=False)

     repetition   recoding        learning  prop_source  prop_target  \
0             0       coot    unsupervised          1.0         0.00   
1             0     jdcoot    unsupervised          1.0         0.00   
2             0     jdcoot  semisupervised          1.0         0.01   
3             0       coot  semisupervised          1.0         0.01   
4             0  reference  semisupervised          1.0         0.01   
..          ...        ...             ...          ...          ...   
315           9       coot         partial          0.5         0.20   
316           9  reference         partial          0.5         0.20   
317           9     jdcoot         partial          0.5         0.40   
318           9       coot         partial          0.5         0.40   
319           9  reference         partial          0.5         0.40   

     pure_source  test_source  pure_target  test_target  
0       1.000000     1.000000     0.235119     0.180432  
1       1.000000   

In [56]:
import pandas as pd

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

In [68]:
df_summary

,recoding,learning,prop_source,prop_target,pure_source_mean,pure_source_var,test_source_mean,test_source_var,pure_target_mean,pure_target_var,test_target_mean,test_target_var
0,coot,partial,0.5,0.01,0.223810,0.004025,0.092503,0.000319,0.476577,0.020816,0.471283,0.025641
1,coot,partial,0.5,0.05,0.398810,0.016338,0.172935,0.051504,0.632500,0.005050,0.623761,0.003843
2,coot,partial,0.5,0.10,0.346429,0.013281,0.461626,0.109756,0.645875,0.013451,0.666201,0.000688
3,coot,partial,0.5,0.20,0.455952,0.007253,0.748920,0.000550,0.578810,0.009579,0.699111,0.001020
4,coot,partial,0.5,0.40,0.447024,0.006554,0.785642,0.000724,0.559406,0.015805,0.790978,0.000396
5,coot,semisupervised,1.0,0.01,1.000000,0.000000,1.000000,0.000000,0.469069,0.011080,0.458196,0.013779
6,coot,semisupervised,1.0,0.05,1.000000,0.000000,1.000000,0.000000,0.676562,0.001474,0.668742,0.000560
7,coot,semisupervised,1.0,0.10,1.000000,0.000000,1.000000,0.000000,0.685149,0.003326,0.682211,0.000566
8,coot,semisupervised,1.0,0.20,1.000000,0.000000,1.000000,0.000000,0.688848,0.003578,0.747522,0.000451
9,coot,semisupervised,1.0,0.40,1.000000,0.000000,1.000000,0.000000,0.663861,0.006042,0.835705,0.000262


In [57]:
df_summary.to_excel("results_mean.xlsx", index=False)

In [ ]:
alpha_grid = np.linspace(0, 1, 11)  # ex: [0.0, 0.1, ..., 1.0]


best_alpha = None
best_score = -np.inf  # ou +np.inf si MSE

numRepetitions_alpha = 5  

for a in alpha_grid:
    scores = []

    for repe in range(numRepetitions_alpha):
        XS1, yS1, XStest, yStest = get_data_2(XtotS, YtotS, nbtrain, nbtest, repe)
        XT1, yT1, XTtest, yTtest = get_data_2(XtotT, YtotT, nbtrain, nbtest, repe)

        S = pd.DataFrame(np.c_[XS1, yS1],
                         columns=[f'X{i}' for i in range(XS1.shape[1])] + ['Z'])
        T = pd.DataFrame(np.c_[XT1, yT1],
                         columns=[f'X{i}' for i in range(XT1.shape[1])] + ['Z'])
        S_test = pd.DataFrame(np.c_[XStest, yStest],
                              columns=[f'X{i}' for i in range(XStest.shape[1])] + ['Z'])
        T_test = pd.DataFrame(np.c_[XTtest, yTtest],
                              columns=[f'X{i}' for i in range(XTtest.shape[1])] + ['Z'])

        _, _, _, test_target = discrete_unsupervised_jdcoot(
            S, T, S_test, T_test, alpha=a
        )

        scores.append(test_target)

    mean_score = np.mean(scores)

    if mean_score > best_score:
        best_score = mean_score
        best_alpha = a

print("Best alpha:", best_alpha)
print("Best score:", best_score)


0.9

In [10]:
import math
results = []
numRepetitions = 10
a = 0.5

prop_target_values_s = [0.01, 0.05, 0.1, 0.2, 0.4]
prop_target_values_p = [0.01, 0.05, 0.1, 0.2, 0.4]

from jdcoot.scenario import generate_data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1

for repe in range(numRepetitions):
    print("num repe :", repe + 1)
    data = generate_data()
    # =========================================================
    # UNSUPERVISED
    # =========================================================

    # COOT (sans alpha)
    pure_source, pure_target, test_source, test_target = continuous_unsupervised_coot(*data)
    print(test_target)
    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

    # JDCOOT
    pure_source, pure_target, test_source, test_target = continuous_unsupervised_jdcoot(*data)
    print(test_target)
    results.append({
        "repetition": repe,
        "recoding": "jdcoot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

num repe : 1
Delta:       0.2181103 	 Loss:       1.9689301
Delta:       0.0004506 	 Loss:       1.7861124
Delta:       0.0000000 	 Loss:       1.7861124
converged at iter  2
630.4723091917492
Delta:       0.2181103 	 Loss:       1.9689301
Delta:       0.0004506 	 Loss:       1.7861124
Delta:       0.0000000 	 Loss:       1.7861124
converged at iter  2
Delta: 0.21892210025777573 	  Loss: 1.9149239409673964 	 Accuracy: 583.194580078125
Delta: 0.22515493446456936 	  Loss: 1.8863706417309696 	 Accuracy: 677.1507568359375
Delta: 0.00029322244045484034 	  Loss: 1.8844867713854834 	 Accuracy: 688.86962890625
Delta: 8.101430325562712e-05 	  Loss: 1.8843320178803142 	 Accuracy: 682.8627319335938
Delta: 8.852923821495225e-05 	  Loss: 1.8843806689337081 	 Accuracy: 701.5531005859375
Delta: 0.14150326320710901 	  Loss: 1.8841414761789097 	 Accuracy: 685.6490478515625
Delta: 8.6589936591149e-05 	  Loss: 1.8842506951534017 	 Accuracy: 689.4417724609375
Delta: 7.393309179188639e-05 	  Loss: 1.884289

In [11]:
df_summary

,recoding,learning,prop_source,prop_target,pure_source_mean,pure_source_var,test_source_mean,test_source_var,pure_target_mean,pure_target_var,test_target_mean,test_target_var
0,coot,unsupervised,1,0,0.0,0.0,0.000000,0.000000,694.654605,0.000000,630.756940,0.499914
1,jdcoot,unsupervised,1,0,0.0,0.0,412.270749,0.101959,681.577705,36.828651,738.784846,56.236471
